# Outils d'analyse des données trafic

## Nettoyage et agrégation par flag

In [2]:
import polars as pl
from datetime import datetime

def process_siredo_traffic(df: pl.DataFrame) -> pl.DataFrame:
    """
    Traite un DataFrame de données individuelles SIREDO (spm_bs_individuelle) :
    1. Ajout de flags de qualité (présence vs vitesse valide).
    2. Agrégation par station et par pas de temps (6 minutes).
    3. Calcul propre des débits et des vitesses moyennes.
    """
    
    # Étape 1 : Application des flags de qualification
    df_flagged = df.with_columns([
        # Flag 1 : Présence physique valide (exclut les erreurs fatales de capteurs : 
        # vitesses aberrantes hors limites physiques absolues ou longueurs négatives/nulles)
        (
            (pl.col("vitesse") >= 0) & 
            (pl.col("vitesse") <= 200) & 
            (pl.col("longueur") > 0)
        ).alias("is_valid_presence"),
        
        # Flag 2 : Vitesse exploitable pour les analyses statistiques 
        # (exclut les excès extrêmes ou les artefacts de couplage de boucles)
        (
            (pl.col("vitesse") >= 10) & 
            (pl.col("vitesse") <= 160)
        ).alias("is_valid_speed")
    ])
    
    # Étape 2 : Troncature temporelle et agrégation
    df_aggregated = (
        df_flagged
        .with_columns(
            # Découpage temporel standard SIREDO (ex: pas de 6 minutes)
            intervalle = pl.col("timestamp").dt.truncate("6m")
        )
        .group_by(["id_station", "intervalle"])
        .agg([
            # Débit brut (tous les enregistrements reçus)
            pl.len().alias("debit_brut"),
            
            # Débit réel/valide : on compte tous les véhicules ayant physiquement franchi 
            # la station (on ne supprime pas le véhicule du débit sous prétexte qu'il roule vite)
            pl.col("is_valid_presence").sum().alias("debit_valide"),
            
            # Vitesse moyenne calculée uniquement sur les véhicules dont la vitesse est valide
            pl.when(pl.col("is_valid_speed"))
              .then(pl.col("vitesse"))
              .mean()
              .alias("vitesse_moyenne"),
              
            # Indicateur de qualité : taux de rejet des vitesses sur l'intervalle
            (
                1.0 - (pl.col("is_valid_speed").sum() / pl.col("is_valid_presence").sum().clip(lower_bound=1))
            ).alias("taux_rejet_vitesse")
        ])
        .sort(["id_station", "intervalle"])
    )
    
    return df_aggregated


# ==========================================
# Exemple d'utilisation avec des données simulées
# ==========================================
if __name__ == "__main__":
    # Simulation d'un petit jeu de données SIREDO individuel
    data = {
        "id_station": ["STA_01", "STA_01", "STA_01", "STA_01", "STA_01"],
       "timestamp": [
            datetime(2026, 10, 14, 8, 0, 2),
            datetime(2026, 10, 14, 8, 0, 15),
            datetime(2026, 10, 14, 8, 2, 10),
            datetime(2026, 10, 14, 8, 5, 45),
            datetime(2026, 10, 14, 8, 6, 12),
        ],
        "vitesse": [135.0, 145.0, -1.0, 185.0, 90.0],  # -1 (erreur capteur), 185 (aberrant/technique)
        "longueur": [4.5, 16.2, 4.0, 5.0, 4.2],         # Véhicule 2 est un Poids Lourd
        "classe": ["VL", "PL", "VL", "VL", "VL"]
    }
    
    df_brut = pl.DataFrame(data)
    
    print("--- Données Brutes ---")
    print(df_brut)
    
    df_resultat = process_siredo_traffic(df_brut)
    
    print("\n--- Données Agrégées et Nettoyées (Pas de 6 min) ---")
    print(df_resultat)

--- Données Brutes ---
shape: (5, 5)
┌────────────┬─────────────────────┬─────────┬──────────┬────────┐
│ id_station ┆ timestamp           ┆ vitesse ┆ longueur ┆ classe │
│ ---        ┆ ---                 ┆ ---     ┆ ---      ┆ ---    │
│ str        ┆ datetime[μs]        ┆ f64     ┆ f64      ┆ str    │
╞════════════╪═════════════════════╪═════════╪══════════╪════════╡
│ STA_01     ┆ 2026-10-14 08:00:02 ┆ 135.0   ┆ 4.5      ┆ VL     │
│ STA_01     ┆ 2026-10-14 08:00:15 ┆ 145.0   ┆ 16.2     ┆ PL     │
│ STA_01     ┆ 2026-10-14 08:02:10 ┆ -1.0    ┆ 4.0      ┆ VL     │
│ STA_01     ┆ 2026-10-14 08:05:45 ┆ 185.0   ┆ 5.0      ┆ VL     │
│ STA_01     ┆ 2026-10-14 08:06:12 ┆ 90.0    ┆ 4.2      ┆ VL     │
└────────────┴─────────────────────┴─────────┴──────────┴────────┘

--- Données Agrégées et Nettoyées (Pas de 6 min) ---
shape: (2, 6)
┌────────────┬──────────────┬────────────┬──────────────┬─────────────────┬────────────────────┐
│ id_station ┆ intervalle   ┆ debit_brut ┆ debit_valide ┆ vit